# 05. 정량 평가 (45개 질의 + 베이스라인 비교)

정형 수치 26개 + 비정형 서술 19개 = 총 45개 질의로 시스템 성능을 평가하고, 두 베이스라인과 3-way 비교합니다.

**소요 시간**: 약 10-15분 (HCX API 호출 한도 주의)

## 5.1 환경 설정

In [ ]:
import os, sys, json
from google.colab import userdata
sys.path.insert(0, '/content/repo')

# 환경변수
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['CLOVA_API_KEY'] = userdata.get('CLOVA_API_KEY')

# 결과 저장 디렉토리
RESULTS_DIR = '/content/eval_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 5.2 본 시스템 평가

In [ ]:
from eval.run_evaluation import run_full_evaluation

# 45개 질의 실행
results = run_full_evaluation(
    output_path=f'{RESULTS_DIR}/pipeline_eval_results.json',
)

print(f"\n본 시스템 평가 결과:")
print(f"  Fact (정형 수치):  {results['fact_correct']}/{results['fact_total']} = {results['fact_accuracy']*100:.1f}%")
print(f"  Narrative (서술): {results['narrative_correct']}/{results['narrative_total']} = {results['narrative_accuracy']*100:.1f}%")
print(f"  전체 평균 응답시간: {results['avg_elapsed']:.1f}초")

## 5.3 베이스라인 시스템 구축

공정 비교를 위해 두 베이스라인을 같은 데이터로 색인:

In [ ]:
# Baseline A: 단순 본문 RAG (XBRL 미사용)
from baseline.baseline_a_build import build_baseline_a

build_baseline_a(
    raw_dir='/content/data/raw',
    output_dir='/content/baseline/a',
)
print("베이스라인 A 구축 완료 (단순 본문 RAG)")

In [ ]:
# Baseline B: 마크다운 표 보존 RAG (중간보고서 시스템)
from baseline.baseline_b_build import build_baseline_b

build_baseline_b(
    raw_dir='/content/data/raw',
    output_dir='/content/baseline/b',
)
print("베이스라인 B 구축 완료 (마크다운 표 보존)")

## 5.4 베이스라인 평가 (동일 26개 정형 질의)

In [ ]:
from baseline.baseline_run import evaluate_baseline

# Baseline A
result_a = evaluate_baseline(
    baseline_dir='/content/baseline/a',
    output_path=f'{RESULTS_DIR}/baseline_a_results.json',
)

# Baseline B
result_b = evaluate_baseline(
    baseline_dir='/content/baseline/b',
    output_path=f'{RESULTS_DIR}/baseline_b_results.json',
)

print(f"\n 베이스라인 평가 결과:")
print(f"  A (단순 본문 RAG):     {result_a['correct']}/{result_a['total']} = {result_a['accuracy']*100:.1f}%")
print(f"  B (마크다운 표 RAG):   {result_b['correct']}/{result_b['total']} = {result_b['accuracy']*100:.1f}%")

## 5.5 3-way 비교 결과 (본 연구의 핵심)

In [ ]:
from baseline.compare_results import compare_three_way

comparison = compare_three_way(
    pipeline_path=f'{RESULTS_DIR}/pipeline_eval_results.json',
    baseline_a_path=f'{RESULTS_DIR}/baseline_a_results.json',
    baseline_b_path=f'{RESULTS_DIR}/baseline_b_results.json',
)

print("=" * 60)
print("  베이스라인 3-way 비교 (정형 수치 26개)")
print("=" * 60)
print(f"  A: 단순 본문 RAG          {comparison['baseline_a']*100:>5.1f}%")
print(f"  B: 마크다운 표 보존 RAG    {comparison['baseline_b']*100:>5.1f}%")
print(f"  ★ 본 시스템 (XBRL+RAG)    {comparison['pipeline']*100:>5.1f}%")
print()
print(f"  본 시스템 / A:  {comparison['pipeline']/comparison['baseline_a']:.0f}배 향상")
print(f"  본 시스템 / B:  {comparison['pipeline']/comparison['baseline_b']:.1f}배 향상")

## 5.6 결과 시각화

In [ ]:
# 그림 4-2 생성 (베이스라인 3-way 비교 막대 차트)
!python /content/repo/scripts/generate_figures.py
print("\n docs/figures/ 폴더에 그림 5개 생성")

## 5.7 다음 단계

FastAPI + Streamlit 웹 애플리케이션을 ngrok으로 외부에 노출합니다.

→ **06_deploy_web.ipynb** 로 진행